# 02 Data transformation
Input: `flows_clean.csv`. Output: typed `flows_transformed.csv`.

In [ ]:
import sys, ipaddress
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
from src import config as cfg

df = pd.read_csv(cfg.DATA_PROCESSED / "flows_clean.csv")
df[cfg.TIME] = pd.to_datetime(df[cfg.TIME])
df[cfg.PROTOCOL] = df[cfg.PROTOCOL].map(lambda v: (cfg.PROTOCOL_CODE_MAP or {}).get(int(v), f"PROTO_{int(v)}") if pd.notna(v) else pd.NA).astype("string")
for c in [cfg.SRC_PORT, cfg.DST_PORT]: df[c] = pd.to_numeric(df[c], errors="coerce").round().astype("Int64")

for c in [cfg.SRC_IP, cfg.DST_IP]:
    df[c] = df[c].map(lambda v: str(ipaddress.IPv4Address(int(v))) if pd.notna(v) else pd.NA).astype("string")
match = np.isclose(df[cfg.BYTES_PER_PACKET], df[cfg.BYTES_SENT] / df[cfg.PACKETS], equal_nan=False).mean()
print(f"{cfg.BYTES_PER_PACKET} == {cfg.BYTES_SENT} / {cfg.PACKETS}: {match:.1%} of rows")
skewness = df[cfg.NUMERIC_COLS].skew(numeric_only=True).sort_values(ascending=False).to_frame("skewness")
display(df.head())
display(skewness)
df.to_csv(cfg.DATA_PROCESSED / "flows_transformed.csv", index=False)
print("Saved", df.shape)